# Predict Customer Churn - Advanced LightGBM Multi-Seed Ensemble

This notebook applies our most advanced Multi-Seed variance-reduction architecture to **LightGBM**, which has historically held our strongest baseline scores in the **"Predict Customer Churn"** Kaggle competition.

**Architecture Overview:**
1.  **Rigorous Tuning:** Performs 100 Trials using Optuna. Crucially, the objective function evaluates internally across a 5-Fold Stratified CV (`lgb.cv`), identifying hyperparameters that generalize reliably across folds rather than exclusively fitting to a stationary validation split.
2.  **Multi-Seed Ensembling:** After the optimal parameters are resolved, the notebook builds **50 distinct LightGBM models** (10 Folds × 5 explicit random seeds) and averages their test set predictions to form a highly resilient final submission.

**Hardware Switch:** Employs GPU-safe bounds natively compatible with Kaggle. Make sure **"GPU"** is selected in your Kaggle notebook settings.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
import warnings
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Reduce output noise

# --- 1. SETTINGS & PATHS ---
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle environment.")
    DATA_DIR = '/kaggle/input/playground-series-s6e3/'
else:
    print("Running locally.")
    DATA_DIR = '../data/'

# Configuration parameters
RUN_TUNING = True   # Set to True to run the rigorous 100 Trial sweep
N_TRIALS = 100

# Multi-Seed Ensemble settings
SEEDS = [42, 2024, 777, 1337, 999]
N_SPLITS = 10 
TOTAL_MODELS = len(SEEDS) * N_SPLITS

## 2. Load & Preprocess Data
Utilizing clean, raw data features along with explicit label encoding to let LightGBM's tree algorithms parse the underlying boundaries naturally without manual math-feature overfitting.

In [ ]:
print(f"Loading data from {DATA_DIR}...")
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

# Target variable formatting
target = 'Churn'
if train_df[target].dtype == 'object':
    train_df[target] = train_df[target].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# Combine for unified preprocessing
train_df['is_train'] = 1
test_df['is_train'] = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', target, 'is_train']]
categorical_features = []

print("Encoding Categorical values...")
for col in features:
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

# Fill missing numerical features
num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Split back
train_encoded = df[df['is_train'] == 1].reset_index(drop=True)
test_encoded = df[df['is_train'] == 0].reset_index(drop=True)

X = train_encoded[features]
y = train_encoded[target]
X_test = test_encoded[features]

print(f"Final Train shape: {X.shape}, Test shape: {X_test.shape}")

## 3. High-Fidelity Optuna Hyperparameter Tuning using `lgb.cv`
Testing 100 dynamic parameter architectures mapped over an internal Stratified 5-Fold routine to guarantee parameters are fully optimized to out-of-fold generalization.

In [ ]:
# Preallocate standard base parameters
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'feature_pre_filter': False  # Prevents crash on Kaggle instances when tuning min_data_in_leaf
}

# Enable GPU if in Kaggle
if IS_KAGGLE:
    best_params['device_type'] = 'gpu'

# Load full data into LightGBM mapping
dtrain = lgb.Dataset(X, label=y, categorical_feature=categorical_features, free_raw_data=False, params={'feature_pre_filter': False})

if RUN_TUNING:
    print(f"\nStarting {N_TRIALS}-Trial Exhaustive Optuna Tuning for LightGBM...")
    print("Using an internal 5-Fold 'lgb.cv' for generalization checks per trial.")
    
    def objective(trial):
        params = best_params.copy()
        
        # Enforcing heavily on `min_child_samples` for regularization safely instead.
        max_depth = trial.suggest_int('max_depth', 3, 10)
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, max(16, min(127, 2**max_depth - 1))), # Dynamic capping directly prevents root leaf overflow exceptions
            'max_depth': max_depth,
            'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        })

        try:
            # Execute Native CV Search
            cv_result = lgb.cv(
                params,
                dtrain,
                num_boost_round=1500,
                nfold=5,
                stratified=True,
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
                seed=42,
                return_cvbooster=False
            )
            
            # Retrieve max score from valid 'auc-mean' mapping explicitly inside the exception handling
            best_auc = max(cv_result['valid auc-mean'])
            return best_auc
        except lgb.basic.LightGBMError as e:
            print(f"\n[Warning] Trial pruned due to LightGBM Environment Crash: {e}")
            raise optuna.TrialPruned()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    print(f"\n[OPTUNA COMPLETE] Best Internal 5-Fold CV AUC: {study.best_value:.5f}")
    print("Optimal Parameters found:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
        best_params[k] = v
        
else:
    print("\nSkipping Optuna. Loading known generalized baseline LightGBM defaults...")
    precomputed_params = {
        'learning_rate': 0.01811,
        'num_leaves': 84,
        'max_depth': 10,
        'feature_fraction': 0.6537,
        'min_child_samples': 20,
        'lambda_l1': 0.01859,
        'lambda_l2': 0.00646,
    }
    best_params.update(precomputed_params)

## 4. Multi-Seed Ensemble (50 LightGBM Models)
We train across 10 independent stratified subset folds looping through 5 overarching randomized seeds, generating 50 independent LightGBM predictors. This brute-forces massive stability ensuring Kaggle private LB safety.

In [ ]:
print(f"\nInitiating Multi-Seed Massive LightGBM Ensemble")
print(f"Architecture: {len(SEEDS)} Seeds × {N_SPLITS}-Folds = {TOTAL_MODELS} Total Models")

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n===> LightGBM Seed Sequence {seed} ({seed_idx + 1}/{len(SEEDS)}) <===")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    seed_oof = np.zeros(len(X))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_f, y_train_f = X.iloc[train_idx], y.iloc[train_idx]
        X_valid_f, y_valid_f = X.iloc[val_idx], y.iloc[val_idx]
        
        lgbm_train = lgb.Dataset(X_train_f, label=y_train_f, categorical_feature=categorical_features, free_raw_data=False, params={'feature_pre_filter': False})
        lgbm_valid = lgb.Dataset(X_valid_f, label=y_valid_f, reference=lgbm_train, categorical_feature=categorical_features, free_raw_data=False, params={'feature_pre_filter': False})
        
        model_fold = lgb.train(
            best_params,
            lgbm_train,
            num_boost_round=3000, 
            valid_sets=[lgbm_valid],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, verbose=False)
            ]
        )
        
        # Predict local validation
        fold_val_preds = model_fold.predict(X_valid_f, num_iteration=model_fold.best_iteration)
        seed_oof[val_idx] = fold_val_preds
        global_oof_sum[val_idx] += fold_val_preds
        
        # Accumulate test predictions across all 50 models
        fold_test_preds = model_fold.predict(X_test, num_iteration=model_fold.best_iteration)
        test_preds_sum += fold_test_preds
        
        print(f"   Fold {fold+1:02d} | Valid AUC: {roc_auc_score(y_valid_f, fold_val_preds):.5f}")
        
    # Review specific seed aggregate stability
    seed_auc = roc_auc_score(y, seed_oof)
    print(f"   >>> Internal OOF AUC for Seed {seed}: {seed_auc:.5f}")

# Calculate final blended validation metric across ALL 50 models natively averaging
global_oof_avg = global_oof_sum / len(SEEDS)
final_oof_auc = roc_auc_score(y, global_oof_avg)

print("\n======================================================")
print(f"FINAL AVERAGED MULTI-SEED ENSEMBLE OOF AUC: {final_oof_auc:.5f}")
print("======================================================")

## 5. Generate Submission File

In [ ]:
print(f"\nBlending test outputs across all {TOTAL_MODELS} trained LightGBM iterations...")
final_test_preds = test_preds_sum / TOTAL_MODELS

sub[target] = final_test_preds
sub.to_csv('submission_lightgbm_multiseed_100t.csv', index=False)

print("Submission generated: 'submission_lightgbm_multiseed_100t.csv'")
display(sub.head())